In [1]:

import numpy as np
import networkx as nx
from itertools import combinations
from time import time
import pickle
from argparse import Namespace
from scipy.optimize import minimize, OptimizeResult, basinhopping

from qiskit import QuantumCircuit, generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.converters import dag_to_circuit, circuit_to_dag
from qiskit.transpiler import Layout
from qiskit.circuit import Parameter

from qopt_best_practices.transpilation.qaoa_construction_pass import QAOAConstructionPass

from qiskit_aer import AerSimulator
from qiskit_aer.backends.backendconfiguration import AerBackendConfiguration
from qiskit_aer.primitives import SamplerV2 as Sampler

# from qiskit_qaoa.utils.qaoa_circuit_utils import get_mixer_operator, state_prep
from qiskit_qaoa.utils.transpiler_passes import ExtendedSwapStrategy
from qiskit_qaoa.utils.pass_managers import get_hubo_pass_manager
from qiskit_qaoa.utils.string_utils import evaluate_sparse_pauli_samples


In [45]:



def print_circuit_info(qc, circuit_name):
    print(f'{circuit_name} has {qc.count_ops().get("cz", 0) + qc.count_ops().get("rzz", 0) + qc.count_ops().get("cx", 0)} 2Q gates \
    and {qc.depth(lambda instr: len(instr.qubits) > 1)} 2Q depth')
    
    
args = Namespace(**dict(
    filename='test_N3_W4',
    extra=1,
    swap_depth=10,
    fraction_four=0.0,
    fraction_six=1.0,
    times_to_keep=(0,1),
    copy_numbers=[2,1,1],
    coupling_map='grid',
    reps=4,
    method='Powell',
    shots=10000,
    init='random',
    alpha=0.25,
    nodes=3,
    time=4
))

filename: str = args.filename
p: int = args.reps
shots: int = args.shots
init_type: str = args.init
swap_depth: int = args.swap_depth
N: int = args.nodes
T: int = args.time
n = int(np.ceil(np.log2(2*N+1)))

seed = 1
rng = np.random.default_rng()

basis_gates=["sx", "x", "rz", "rzz", "cz", "id", "swap", "cx", "h"]


basepath = '/lustre/scratch127/qpg/jc59/hubo/'
filename = 'simulation.{}.compilation.{}.extra{}.times{}.four{}.six{}'.format(
    args.coupling_map,
    args.filename,
    args.extra,
    ''.join([str(t) for t in args.times_to_keep]),
    args.fraction_four,
    args.fraction_six
)
results_file = basepath + filename + '.pkl'

with open(results_file, 'rb') as f:
    data = pickle.load(f)
compiled_hamiltonian: SparsePauliOp = data['compiled_hamiltonian']
full_hamiltonian: SparsePauliOp = data['full_hamiltonian']
layout: Layout = data[swap_depth]

num_qubits: int = full_hamiltonian.num_qubits if full_hamiltonian.num_qubits is not None else max(layout.get_physical_bits().keys())

if args.coupling_map == 'line':
    extended_swap_strat = ExtendedSwapStrategy.from_line(list(range(num_qubits)), num_swap_layers=1000)
elif args.coupling_map == 'grid':
    extended_swap_strat = ExtendedSwapStrategy.from_grid(n, T)
else:
    raise Exception('Invalid coupling map type')

num_physical_qubits = extended_swap_strat._num_vertices
coupling_map = extended_swap_strat._coupling_map

backend_options = dict(
    method='statevector',
    device='CPU',
    precision='single',
    basis_gates=basis_gates,
)


config = AerSimulator._DEFAULT_CONFIGURATION
config["n_qubits"] = num_physical_qubits
config["basis_gates"] = basis_gates
config = AerBackendConfiguration.from_dict(config)
backend = AerSimulator(configuration=config, coupling_map=extended_swap_strat._coupling_map, **backend_options)
backend.set_option("n_qubits", num_physical_qubits)
sampler = Sampler().from_backend(backend)

donor_qc = QuantumCircuit(num_qubits)
remapped_full_hamiltonian = full_hamiltonian.apply_layout([layout.get_virtual_bits()[donor_qc.qubits[i]] for i in range(num_qubits)], num_physical_qubits)


coupling_map_edge = list(coupling_map)
physical_qubits = list(coupling_map.physical_qubits)
dual_coupling_map = nx.Graph()

for qubit in physical_qubits:
    edges = [edge for edge in coupling_map_edge if edge[0]==qubit]
    for edge1, edge2 in combinations(edges, 2):
        dual_coupling_map.add_edge(tuple(sorted(edge1)), tuple(sorted(edge2)))
edge_colouring = nx.greedy_color(dual_coupling_map, interchange=True)


pm = get_hubo_pass_manager(extended_swap_strat, swap_depth, args.extra)

cost_qc = QuantumCircuit(num_physical_qubits)
cost_qc.append(PauliEvolutionGate(compiled_hamiltonian, time=Parameter("c")), [layout.get_virtual_bits()[donor_qc.qubits[i]] for i in range(num_physical_qubits)])
tcost_qc = pm.run(cost_qc)

print_circuit_info(tcost_qc, 'Transpiled cost hamiltonian circuit')
print(tcost_qc.count_ops())
print(f'Cost hamiltonian circuit has {tcost_qc.num_qubits} qubits')


sp = None
sp = QuantumCircuit(num_qubits)
for i in [5, 10, 11]:
    sp.x(i)
mixer = None
    
    
construction_pass = QAOAConstructionPass(p, init_state=sp, mixer_layer=mixer)
qaoa_circ = dag_to_circuit(construction_pass.run(circuit_to_dag(tcost_qc)))

# Now transpile to basis gates
generic_pm = generate_preset_pass_manager(optimization_level=3, backend=backend, basis_gates=basis_gates)
init  = generic_pm.init
init.remove(3)
generic_pm.init = init
generic_pm.layout = None
t_qaoa_circ = generic_pm.run(qaoa_circ)

print_circuit_info(t_qaoa_circ, 'QAOA circuit')
print(t_qaoa_circ.count_ops())
print(f'QAOA circuit has {t_qaoa_circ.num_qubits} qubits')


qaoa_depth = len(t_qaoa_circ.parameters) // 2
if init_type == 'ramp':
    t = 0.7 * p
    betas = np.linspace(
        (1 / p) * (t * (1 - 0.5 / p)), (1 / p) * (t * 0.5 / p), p
    )
    gammas = betas[::-1]
    init_params = betas.tolist() + gammas.tolist()
elif init_type == 'warm':
    init_params = [0.56679859, 0.35556051, 0.4503177,  0.20867354, 0.48058088, 0.42463428, 0.40800271, 0.39104565]
else:
    init_params = rng.uniform(0, 1, qaoa_depth).tolist() + rng.uniform(0, 1, qaoa_depth).tolist()
print(f'Init: {init_params}')


print(f'Noise model: {getattr(sampler._backend.options, "noise_model", "Ideal noise")}')

history = []
alpha = 0.25

def cvar(energies, alpha=1.0):
    sorted_energies = sorted(energies)
    end_idx = int(alpha * len(energies))
    return np.sum(sorted_energies[0:end_idx]) / end_idx


def objective(x: np.ndarray):
    start = time()
    assigned_circuit = t_qaoa_circ.assign_parameters(x, inplace=False)
    sampler_job = sampler.run([assigned_circuit], shots=shots)
    sampler_result = sampler_job.result()
    counts = sampler_result[0].data.c.get_counts()
    sampling_time = time() - start
    start = time()
    energies = []
    evals = evaluate_sparse_pauli_samples(counts.keys(), remapped_full_hamiltonian)
    energies = [count * [evals[idx]] for idx, count in enumerate(counts.values())]
    flat_energies = [x for xs in energies for x in xs]
    total_energy = cvar(flat_energies, alpha)

    classical_post_process_time = time() - start
    history.append((sampling_time, total_energy, x.tolist(), counts, classical_post_process_time))
    return total_energy


def callback(intermediate_result: OptimizeResult):
    print(f'Current params: {intermediate_result.x}. Current func value: {intermediate_result.fun}')
    if intermediate_result.fun == -1:
        raise StopIteration
    

def callback_cobyla(xk: np.ndarray):
    print(f'Current params: {xk}.')
    

def callback_basinhopping(x: np.ndarray, f: float, accept: bool):
    print(f'Current params: {x}. Current func value: {f}')


Max layers needed to apply swap decompose: 8
Gates we cannot directly implement: 0
[]
Transpiling accumulator
Transpiled cost hamiltonian circuit has 304 2Q gates     and 222 2Q depth
OrderedDict([('cx', 304), ('rz', 122), ('swap', 32)])
Cost hamiltonian circuit has 12 qubits


/tmp/ipykernel_1916038/834704472.py:121: UserWarning: Providing `coupling_map` and/or `basis_gates` along with `backend` is not recommended, as this will invalidate the backend's gate durations and error rates.
  generic_pm = generate_preset_pass_manager(optimization_level=3, backend=backend, basis_gates=basis_gates)


QAOA circuit has 1216 2Q gates     and 888 2Q depth
OrderedDict([('cx', 1216), ('rz', 632), ('swap', 128), ('sx', 96), ('measure', 12), ('x', 3)])
QAOA circuit has 12 qubits
Init: [0.5435497681769347, 0.8144763574011553, 0.3504272329697611, 0.25515262374726866, 0.13632064140247302, 0.041062191326261654, 0.5374843922150956, 0.1375190314568122]
Noise model: None


In [47]:
t_qaoa_circ.draw(fold=-1)

global phase: 226.19467105846533 - 16.375*γ[0] - 16.375*γ[1] - 16.375*γ[2] - 16.375*γ[3]
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      ┌─────────┐       ┌────┐       ┌────────────────┐      ┌────┐     ┌──────────┐                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   

In [48]:
evaluate_sparse_pauli_samples(['000010101000'], full_hamiltonian)

array([0.])

In [ ]:
opt = '000010101000'[::-1]
remapped_opt = [opt[donor_qc.find_bit(layout.get_physical_bits()[i]).index] for i in range(num_qubits)]
remapped_opt = ''.join(remapped_opt[::-1])
print(remapped_opt)
evaluate_sparse_pauli_samples([remapped_opt], remapped_full_hamiltonian)

110000100000


array([0.])

In [56]:
layout

Layout({
9: <Qubit register=(12, "q"), index=0>,
4: <Qubit register=(12, "q"), index=1>,
8: <Qubit register=(12, "q"), index=2>,
11: <Qubit register=(12, "q"), index=3>,
7: <Qubit register=(12, "q"), index=4>,
10: <Qubit register=(12, "q"), index=5>,
6: <Qubit register=(12, "q"), index=6>,
5: <Qubit register=(12, "q"), index=7>,
3: <Qubit register=(12, "q"), index=8>,
2: <Qubit register=(12, "q"), index=9>,
1: <Qubit register=(12, "q"), index=10>,
0: <Qubit register=(12, "q"), index=11>
})

In [60]:
np.array([x for x in '000010101000'[::-1]]) 

array(['0', '0', '0', '1', '0', '1', '0', '1', '0', '0', '0', '0'],
      dtype='<U1')

In [ ]:
[donor_qc.find_bit(layout.get_physical_bits()[i]).index for i in range(num_qubits)]

[11, 10, 9, 8, 1, 7, 6, 4, 2, 0, 5, 3]

In [57]:
[donor_qc.find_bit(layout.get_physical_bits()[i]).index for i in [3,5,7]]

[8, 7, 4]

In [54]:
objective(np.array([0,0,0,0,1,1,1,1]))

np.float64(0.0)

In [39]:
full_hamiltonian.apply_layout([layout.get_virtual_bits()[donor_qc.qubits[i]] for i in range(num_qubits)], num_physical_qubits) == remapped_full_hamiltonian

True

In [ ]:
print(f'Using method: {args.method}.')
if args.method == 'basinhopping':
    result = basinhopping(
        objective, 
        x0=init_params, 
        niter=100,
        minimizer_kwargs=dict(bounds=tuple((0,1) for _ in range(2 * p)),method="Powell",options={"maxiter":100, "maxfev":1000}),
        callback=callback_basinhopping,
        disp=True
    )
else:
    result = minimize(
        objective, 
        x0=init_params, 
        method=args.method, 
        bounds=tuple((0,1) for _ in range(2 * p)), 
        options={"maxiter": 100, "maxfev": 10000, "rhobeg": 0.05, "ftol": 1e-7},
        callback=callback if args.method not in ['SLSQP', 'COBYLA', 'TNC'] else callback_cobyla
    )
    print(result)